# El Niño's atmospheric half: Graphs A and B

**DSC 106 Project 3 checkpoint, exploratory graphs.** Our team studies **El Niño and La Niña** using climate model data. This notebook adds two graphs about the *air* above the ocean:

| Graph | Question | Data used |
|---|---|---|
| **A. The air-pressure seesaw** | When the Pacific Ocean warms, does the air pressure across the Pacific change too? | Sea-level pressure (`psl`) |
| **B. Storm clouds on the move** | Where do the tropics' big rainstorms go during El Niño? | Outgoing heat radiation (`rlut`) |

Together with our teammates' graphs (ocean temperature and rainfall), they tell a chain of cause and effect: **the ocean warms → the air pressure pattern flips (A) → the storms move (B) → rainfall shifts around the world.**

This notebook assumes no background in climate science. Every term is explained when it first appears.

## 0. The background in five minutes

### What are El Niño and La Niña?

Along the equator in the Pacific Ocean, steady winds called the **trade winds** blow from east (South America) to west (Indonesia and Australia). They push warm surface water west, so normally:

- The **western Pacific** (near Indonesia) has the warmest ocean on Earth. Warm water heats the air above it, the air rises, and that builds huge **thunderstorms** with lots of rain.
- The **eastern Pacific** (near Peru and Ecuador) is cooler, because cold water wells up from below. The air there sinks, the skies are mostly clear, and it rarely rains.

Every 2 to 7 years this pattern shifts:

- **El Niño:** the trade winds weaken, warm water spreads east across the Pacific, and the storms follow it east.
- **La Niña:** the opposite. The trade winds get stronger, the east gets even cooler, and the storms stay packed in the west.

Scientists call the whole cycle **ENSO**, short for **E**l **N**iño–**S**outhern **O**scillation. *El Niño* is the ocean part. The *Southern Oscillation* is the air-pressure part, which Graph A shows.

### Why air pressure and clouds?

- **Air pressure** is the weight of the air above a spot. Where air **rises** (over warm water, under storms), pressure at the surface is **lower**. Where air **sinks** (over cool water, under clear skies), pressure is **higher**. When the warm water moves, the pressure pattern moves with it.
- **Storm clouds** can't easily be counted from space, but they can be detected. Every surface sends heat out to space as invisible infrared light. Tall thunderclouds have tops 10–15 km up, where it's about −60 °C, so they send out **much less** heat than a clear warm ocean. Satellites (and climate models) measure this **outgoing longwave radiation**. **Low values mean storm clouds.**

### Key terms used in this notebook

| Term | Meaning |
|---|---|
| **Anomaly** | A value minus its *normal* for that time of year. "Darwin's pressure anomaly is +2 hPa" means Darwin's pressure is 2 hPa higher than a typical month like it. Using anomalies removes the seasons, so we see only unusual changes. |
| **Normal (climatology)** | The average for each calendar month over 1981–2010: an average January, an average February, and so on. This is the same reference period NOAA uses. |
| **Niño 3.4 index** | The average ocean-surface temperature anomaly in a box in the central Pacific (5°S–5°N, 170°W–120°W). It is the standard thermometer for El Niño. |
| **ONI** (Oceanic Niño Index) | The Niño 3.4 index averaged over 3 months to smooth out noise. **≥ +0.5 °C** for 5 months in a row = **El Niño**. **≤ −0.5 °C** for 5 months in a row = **La Niña**. Anything else = **neutral**. |
| **hPa** (hectopascal) | Unit of air pressure. Normal sea-level pressure is about 1013 hPa, and El Niño shifts it by only 1–3 hPa, but those small shifts move whole weather systems. |
| **W/m²** (watts per square meter) | Unit of energy flow, here the heat escaping to space from each square meter. Clear tropical skies send out roughly 280 W/m². Areas under big storms send out under 220 W/m². |
| **Hovmöller diagram** | A heatmap with *place* on one axis and *time* on the other. It shows how a pattern moves over time. |

## 1. The data

### Source

All data comes from **CMIP6**, a worldwide project where climate labs run their models through the same standard experiments. Google hosts a free public copy in the cloud bucket `gs://cmip6`. We read it directly; nothing needs to be downloaded by hand.

Every dataset's address follows the same pattern. For example, the pressure data used in Graph A:

`gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/psl/gn/<version>/`

| Path part | Meaning |
|---|---|
| `CMIP6` / `CMIP` | The project (phase 6) and its core set of experiments |
| `NCAR` | National Center for Atmospheric Research (Boulder, Colorado), the lab that ran the model |
| `CESM2` | Community Earth System Model version 2, NCAR's climate model. It divides the globe into boxes about 1° (~100 km) wide and calculates the physics in each one. |
| `amip` | The experiment: **atmosphere only**. The model doesn't simulate the ocean. Instead it is fed the **real, observed** ocean temperatures for every month from 1950 to 2014. So the El Niños in this data happen in the **real years** (for example 1982–83 and 1997–98), and the model calculates how the air, clouds, and rain respond. |
| `r1i1p1f1` | Which run. NCAR ran this experiment 10 times with tiny differences in starting weather. We use run #1, the same as the rest of our team. |
| `Amon` | **A**tmosphere data, **mon**thly averages |
| `psl` | The variable (see below) |
| `gn` | On the model's own (**n**ative) grid |

### Variables used

| Variable | Full name | What it measures | Stored units → we convert to | Used for |
|---|---|---|---|---|
| `psl` | Sea-level pressure | Weight of the air column, adjusted to sea level so mountains don't distort it | Pa → **hPa** (÷ 100) | Graph A |
| `rlut` | Outgoing longwave radiation at the top of the atmosphere | Heat Earth sends to space as infrared light. Low = cold cloud tops = storms. | **W/m²** (no change) | Graph B |
| `ts` | Surface temperature | Over the ocean, this is the observed sea-surface temperature fed into the model | K → **°C** (− 273.15) | The Niño 3.4 index (both graphs) |
| `sftlf` | Land fraction (table `fx`, fixed in time) | Percent of each grid box that is land | % → 0–1 | Drawing coastlines on Graph A's map |

Each variable has three coordinates: `time` (one value per month, Jan 1950–Dec 2014 = 780 months), `lat` (latitude, −90 to 90, south negative), and `lon` (longitude, 0 to 360 degrees east of Greenwich, so 180 = the date line and 210 = 150°W).

In [1]:
# Run once if these packages aren't installed (Colab, or a new environment).
# In VS Code, installing from the terminal is faster: python -m pip install xarray zarr gcsfs cftime pandas
# %pip install xarray zarr gcsfs cftime pandas

In [2]:
import json, html, time, urllib.request, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML, display

MODEL, MEMBER = "CESM2", "r1i1p1f1"
BASE = ("1981", "2010")                    # "normal" period, same as NOAA and our teammates
META = dict(model=MODEL, experiment="amip", member=MEMBER, years="1950–2014", base="1981–2010")

TAHITI = dict(lat=-17.53, lon=210.43)      # Papeete, Tahiti (149.57°W)
DARWIN = dict(lat=-12.46, lon=130.84)      # Darwin, Australia

OUT = Path("out")
(OUT / "data").mkdir(parents=True, exist_ok=True)

### Opening the data

The bucket has a **catalog**, a spreadsheet with one row per dataset. `open_var` looks up a variable in it and opens it. Opening is *lazy*: only the description is read at first, and the numbers download when we call `.load()`. We also cut each variable down to the region we need before loading.

In [3]:
cat = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")

def open_var(variable, table="Amon", member=MEMBER):
    rows = cat.query("source_id == @MODEL and experiment_id == 'amip' and member_id == @member "
                     "and table_id == @table and variable_id == @variable")
    if rows.empty:
        raise ValueError(f"{MODEL} amip {member} {table}/{variable} not in the catalog")
    path = rows.sort_values("version").iloc[-1].zstore
    print("opening", path)
    da = xr.open_zarr(path, consolidated=True, storage_options={"token": "anon"})[variable]
    return da.assign_coords(lon=da.lon % 360).sortby("lon").sortby("lat")

In [4]:
t0 = time.time()
sst  = open_var("ts").sel(lat=slice(-5, 5), lon=slice(190, 240)).load() - 273.15    # °C, Niño 3.4 box only
psl  = open_var("psl").sel(lat=slice(-40, 40), lon=slice(30, 300)).load() / 100      # hPa, Indian + Pacific
rlut = open_var("rlut").sel(lat=slice(-5, 5), lon=slice(40, 290)).load()             # W/m², equatorial strip
land = open_var("sftlf", table="fx").sel(lat=slice(-40, 40), lon=slice(30, 300)).load() / 100
print(f"Loaded in {time.time() - t0:.0f} s")

t = psl.time.values
print(f"Months: {t[0].year}-{t[0].month:02d} to {t[-1].year}-{t[-1].month:02d} ({psl.sizes['time']})")
print(f"Sea-level pressure range: {float(psl.min()):.0f} to {float(psl.max()):.0f} hPa")
print(f"Outgoing longwave radiation range: {float(rlut.min()):.0f} to {float(rlut.max()):.0f} W/m²")

opening gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/ts/gn/v20190218/


/Users/justinjames/miniforge3/envs/elnino/lib/python3.12/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.2. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
/Users/justinjames/miniforge3/envs/elnino/lib/python3.12/site-packages/google/auth/transport/grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details

opening gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/psl/gn/v20190218/
opening gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/rlut/gn/v20190218/
opening gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/fx/sftlf/gn/v20190218/
Loaded in 19 s
Months: 1950-01 to 2014-12 (780)
Sea-level pressure range: 982 to 1040 hPa
Outgoing longwave radiation range: 126 to 313 W/m²


## 2. The Niño 3.4 index: our El Niño thermometer

Both graphs compare the atmosphere with the ocean, so first we need the ocean signal. These steps match our teammate's notebook exactly, so our El Niño and La Niña months are the same across the whole project:

1. **Anomaly:** subtract each calendar month's 1981–2010 average.
2. **Box average:** average over the Niño 3.4 box, giving boxes near the equator slightly more weight because they cover more area (weight = cos(latitude)).
3. **ONI:** 3-month running mean.
4. **Labels:** El Niño / La Niña only if the ONI stays beyond ±0.5 °C for at least 5 months in a row (NOAA's rule).

In [5]:
def anomalies(da):
    clim = da.sel(time=slice(*BASE)).groupby("time.month").mean("time")
    return (da.groupby("time.month") - clim).drop_vars("month")

def area_mean(da, dims=("lat", "lon")):
    return da.weighted(np.cos(np.deg2rad(da.lat))).mean(list(dims))

def label_phase(values, thresh=0.5, min_len=5):
    s = pd.Series(values)
    phase = pd.Series("Neutral", index=s.index)
    for sign, name in [(1, "El Niño"), (-1, "La Niña")]:
        hit = (sign * s) >= thresh
        run_id = (hit != hit.shift()).cumsum()
        run_len = hit.groupby(run_id).transform("sum")
        phase[hit & (run_len >= min_len)] = name
    return phase

nino34 = area_mean(anomalies(sst))
oni = nino34.rolling(time=3, center=True).mean()
dates = [f"{d.year}-{d.month:02d}" for d in oni.time.values]
phase = label_phase(oni.values).values

print(pd.Series(phase).value_counts().to_string())

Neutral    421
La Niña    211
El Niño    148


## 3. Graph A: the air-pressure seesaw (Southern Oscillation)

### The idea

In the 1920s, the British scientist Gilbert Walker noticed something odd in weather records. When air pressure rose at **Darwin, Australia**, it tended to fall at **Tahiti** in the middle of the Pacific, and vice versa, like a seesaw. He called it the **Southern Oscillation**. Decades later, scientists realized it was the atmosphere's side of El Niño.

- **Normal or La Niña:** warm water and rising air sit in the west (near Darwin), so pressure is **low at Darwin and high at Tahiti**. That difference drives the trade winds from east to west.
- **El Niño:** warm water and rising air move east, so pressure **rises at Darwin and falls at Tahiti**. The difference shrinks or flips, and the trade winds weaken.

So we measure **Tahiti's pressure minus Darwin's pressure** (each as an anomaly). This is a simple version of the **Southern Oscillation Index (SOI)**. It should turn **negative during El Niño** and **positive during La Niña**: a mirror image of the ocean index.

### What the graph shows

| Part | What you see | How to read it |
|---|---|---|
| **Map (top)** | Sea-level pressure averaged over all El Niño months *minus* all La Niña months, for the Indian and Pacific Oceans | **Orange** = pressure is higher during El Niño, **purple** = lower. Look for the seesaw: one color over Australia and Indonesia, the other over the central and eastern Pacific. Tahiti and Darwin are marked. |
| **Upper bars** | The Niño 3.4 index (ocean), one bar per month, 1950–2014 | Up = warmer ocean. Bar color = the month's phase: red El Niño, blue La Niña, gray neutral. |
| **Lower bars** | Tahiti minus Darwin pressure anomaly (atmosphere), same months, same colors | If the seesaw theory is right, **red bars point down here when they point up above**. |
| **Scatter (right)** | One dot per month: ocean index (x) vs pressure difference (y) | A downward-sloping cloud means the two move opposite. **r** (correlation, from −1 to +1) measures how tightly: −1 = perfect mirror, 0 = no relationship. |

**Interaction:** hover the bars to read any month (its dot gets circled in the scatter). **Drag across the bars** to choose a range of years; the scatter fades all other months and recalculates *r* for just that period. Use this to check whether the link holds in every decade or only during big events.

**Design choices:** the ocean and the atmosphere are drawn as two **separate panels with their own axes**, lined up on the same timeline. They use different units (°C and hPa), and putting both on one chart with two y-axes would invite misleading comparisons. Using the **same phase color in both panels** links them without needing to read numbers.

In [6]:
psl_anom = anomalies(psl)

# --- the two stations: nearest model grid box to each town
tahiti = psl_anom.sel(**TAHITI, method="nearest")
darwin = psl_anom.sel(**DARWIN, method="nearest")
soi = (tahiti - darwin).rolling(time=3, center=True).mean()      # 3-month mean, like the ONI
print(f"Tahiti grid box: {float(tahiti.lat):.1f}°, {float(tahiti.lon):.1f}°E   "
      f"Darwin grid box: {float(darwin.lat):.1f}°, {float(darwin.lon):.1f}°E")

# --- the map: El Niño months minus La Niña months
is_nino, is_nina = phase == "El Niño", phase == "La Niña"
diff = (psl_anom.isel(time=np.where(is_nino)[0]).mean("time")
        - psl_anom.isel(time=np.where(is_nina)[0]).mean("time"))
diff_c = diff.coarsen(lat=2, lon=2, boundary="trim").mean().sortby("lat", ascending=False)   # ~2° boxes, north at top
land_c = land.coarsen(lat=2, lon=2, boundary="trim").mean().sortby("lat", ascending=False)
lim = max(0.5, float(np.ceil(np.nanpercentile(np.abs(diff_c), 98) * 2) / 2))

# --- numbers worth quoting
ok = ~np.isnan(oni.values) & ~np.isnan(soi.values)
r = np.corrcoef(oni.values[ok], soi.values[ok])[0, 1]
print(f"Correlation between the ocean index and Tahiti−Darwin pressure: r = {r:.2f}")
print(f"Average Tahiti−Darwin pressure: El Niño months {float(soi[is_nino].mean()):+.2f} hPa, "
      f"La Niña months {float(soi[is_nina].mean()):+.2f} hPa")
print(f"El Niño minus La Niña at Darwin: {float(diff.sel(**DARWIN, method='nearest')):+.2f} hPa, "
      f"at Tahiti: {float(diff.sel(**TAHITI, method='nearest')):+.2f} hPa")

Tahiti grid box: -17.4°, 210.0°E   Darwin grid box: -12.7°, 131.2°E
Correlation between the ocean index and Tahiti−Darwin pressure: r = -0.82
Average Tahiti−Darwin pressure: El Niño months -1.25 hPa, La Niña months +1.24 hPa
El Niño minus La Niña at Darwin: +1.29 hPa, at Tahiti: -1.22 hPa


In [7]:
D3_URL = "https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js"

PAGE = '''<!doctype html><html><head><meta charset="utf-8"><title>__TITLE__</title>
<style>
body{margin:0;padding:12px 10px 16px;font-family:-apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;color:#1f1e1c;background:#fff}
h2{font-size:18px;font-weight:600;margin:0 0 6px}
.sub{font-size:13px;color:#4a4945;margin:0 0 10px;max-width:880px;line-height:1.5}
.controls{font-size:13px;margin:0 0 6px;display:flex;gap:16px;align-items:center;flex-wrap:wrap}
.controls label{cursor:pointer}
.note{font-size:12px;color:#6b6a65;margin:6px 0 0}
.tip{position:absolute;pointer-events:none;background:#fff;border:1px solid #d3d1c7;border-radius:6px;padding:6px 9px;font-size:12px;line-height:1.45;box-shadow:0 2px 8px rgba(0,0,0,.12);opacity:0}
svg text{font-size:11px;fill:#5f5e5a}
svg .ptitle{font-size:13px;font-weight:600;fill:#1f1e1c}
svg .boxlab{font-size:11px;font-weight:600;fill:#1f1e1c}
.axis path,.axis line{stroke:#b4b2a9}
.brush .selection{fill:#1f1e1c;fill-opacity:.12;stroke:#1f1e1c}
</style>__D3__</head><body><div id="root"></div>
<script>const DATA = __DATA__;
__JS__
</script></body></html>'''

_d3_cache = {}
def _d3_tag(inline):
    if not inline:
        return f'<script src="{D3_URL}"></script>'
    if "tag" not in _d3_cache:
        try:   # embed D3 itself so the chart also works where outputs can't load CDNs
            src = urllib.request.urlopen(D3_URL, timeout=15).read().decode("utf-8")
            _d3_cache["tag"] = f"<script>{src}</script>"
        except Exception:
            _d3_cache["tag"] = f'<script src="{D3_URL}"></script>'
    return _d3_cache["tag"]

def build_page(title, js, data, inline):
    page = PAGE.replace("__TITLE__", title).replace("__JS__", js)
    page = page.replace("__DATA__", json.dumps(data, separators=(",", ":"), allow_nan=False))
    return page.replace("__D3__", _d3_tag(inline))

def show_d3(name, title, js, data, height):
    path = OUT / f"{name}.html"
    path.write_text(build_page(title, js, data, inline=False), encoding="utf-8")
    page = build_page(title, js, data, inline=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)   # IPython suggests IFrame; srcdoc keeps it self-contained
        display(HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" '
                     f'style="width:100%;max-width:920px;height:{height}px;border:0"></iframe>'))
    print(f"Saved {path}; open it in a browser to see it full size.")

def clean(a, nd=2):
    # numpy → JSON-friendly lists, with missing values (land, edges) as null
    a = np.asarray(a, dtype=float)
    out = np.round(a, nd).astype(object)
    out[np.isnan(a)] = None
    return out.tolist()

In [8]:
soi_lim = float(np.ceil(np.nanmax(np.abs(soi.values)) + 0.5))
DATA_A = dict(
    meta={**META, "soi_lim": soi_lim},
    map=dict(lat=clean(diff_c.lat.values), lon=clean(diff_c.lon.values), diff=clean(diff_c.values),
             land=clean(land_c.values), n_nino=int(is_nino.sum()), n_nina=int(is_nina.sum()), lim=lim),
    sites=dict(tahiti=TAHITI, darwin=DARWIN),
    time=dates, oni=clean(oni.values), soi=clean(soi.values),
    tahiti=clean(tahiti.values), darwin=clean(darwin.values), phase=list(phase),
)
pd.DataFrame({"date": dates, "oni": DATA_A["oni"], "tahiti_minus_darwin_hpa": DATA_A["soi"],
              "phase": phase}).to_csv(OUT / "data" / "soi.csv", index=False)

In [9]:
GRAPH_A_JS = r"""
// Graph A: the Southern Oscillation (air-pressure seesaw) vs the Niño 3.4 index (D3 v7)
// DATA = {meta, map:{lat (north→south), lon, diff[[...]], land[[...]], n_nino, n_nina, lim},
//         sites:{tahiti:{lat,lon}, darwin:{lat,lon}},
//         time[], oni[], soi[], tahiti[], darwin[], phase[]}
const M = DATA.meta;
const root = d3.select("#root");
root.append("h2").text("When the ocean warms, the air-pressure seesaw tips");
root.append("p").attr("class", "sub").html(
  `<b>Top:</b> how air pressure at sea level differs between El Niño months and La Niña months across the Indian and Pacific Oceans. ` +
  `<b>Bottom:</b> the ocean signal (Niño 3.4 index) and the atmosphere signal (pressure at Tahiti minus pressure at Darwin) for every month, ` +
  `both as departures from their ${M.base} normal and smoothed over 3 months. ` +
  `${M.model} ${M.experiment} run (${M.member}), ${M.years}: real observed ocean temperatures, simulated atmosphere.`
);

const RED = "#e34948", BLUE = "#2a78d6", GRAY = "#c9c6bd";
const phaseColor = p => (p === "El Niño" ? RED : p === "La Niña" ? BLUE : GRAY);
const num = (v, u, d = 2) => (v == null ? "n/a" : (v >= 0 ? "+" : "−") + Math.abs(v).toFixed(d) + (u ? " " + u : ""));
const deg = v => (Math.abs(v) % 1 === 0 ? Math.abs(v) : Math.abs(v).toFixed(1));
const lonFmt = l => { l = ((l % 360) + 360) % 360; return Math.round(l) === 180 ? "180°" : l < 180 ? `${deg(l)}°E` : `${deg(360 - l)}°W`; };
const latFmt = l => (Math.abs(l) < 1e-9 ? "0°" : l > 0 ? `${deg(l)}°N` : `${deg(-l)}°S`);

const W = 880, ml = 52;
const mp = DATA.map, lat = mp.lat, lon = mp.lon, nlat = lat.length, nlon = lon.length;
const dlon = lon[1] - lon[0], dlat = Math.abs(lat[0] - lat[1]);
const pw = W - ml - 20, mapH = Math.round(pw * (nlat * dlat) / (nlon * dlon));
const cw = pw / nlon, ch = mapH / nlat;
const mapTop = 30;
const tTop = mapTop + mapH + 132, tw = 520, th = 118, tgap = 40;
const sLeft = ml + tw + 64, ss = W - sLeft - 16;
const H = tTop + 2 * th + tgap + 44;
const svg = root.append("svg").attr("viewBox", `0 0 ${W} ${H}`).attr("width", W).style("max-width", "100%").style("height", "auto");
const tip = d3.select("body").append("div").attr("class", "tip");

// ---------------- map ----------------
const L = mp.lim;
const pcolor = d3.scaleDiverging([-L, 0, L],
  d3.piecewise(d3.interpolateLab, ["#3d2a7a", "#6b5bc4", "#b9b0e6", "#f0efec", "#f2c38f", "#d9822b", "#8a4a0e"])).clamp(true);
const mg = svg.append("g").attr("transform", `translate(${ml},${mapTop})`);
mg.append("text").attr("y", -10).attr("class", "ptitle")
  .text(`Sea-level pressure in El Niño months minus La Niña months (${mp.n_nino} vs ${mp.n_nina} months)`);
const cells = [];
for (let i = 0; i < nlat; i++) for (let j = 0; j < nlon; j++) cells.push([i, j]);
mg.append("g").selectAll("rect").data(cells).join("rect")
  .attr("x", d => d[1] * cw).attr("y", d => d[0] * ch).attr("width", cw + 0.4).attr("height", ch + 0.4)
  .attr("fill", d => { const v = mp.diff[d[0]][d[1]]; return v == null ? "#f6f5f2" : pcolor(v); });
const coast = d3.contours().size([nlon, nlat]).thresholds([0.5])(mp.land.flat().map(v => (v == null ? 0 : v)));
mg.append("path").attr("d", d3.geoPath(d3.geoTransform({ point(x, y) { this.stream.point(x * cw, y * ch); } }))(coast[0]))
  .attr("fill", "none").attr("stroke", "#1f1e1c").attr("stroke-width", 0.8);
const mx = d3.scaleLinear([lon[0] - dlon / 2, lon[nlon - 1] + dlon / 2], [0, pw]);
const my = d3.scaleLinear([lat[0] + dlat / 2, lat[nlat - 1] - dlat / 2], [0, mapH]);
mg.append("g").attr("class", "axis").attr("transform", `translate(0,${mapH})`)
  .call(d3.axisBottom(mx).tickValues([60, 90, 120, 150, 180, 210, 240, 270].filter(t => t > mx.domain()[0] && t < mx.domain()[1])).tickFormat(lonFmt).tickSizeOuter(0));
mg.append("g").attr("class", "axis")
  .call(d3.axisLeft(my).tickValues([-30, -15, 0, 15, 30].filter(t => t <= my.domain()[0] && t >= my.domain()[1])).tickFormat(latFmt).tickSizeOuter(0));
Object.entries({ Tahiti: DATA.sites.tahiti, Darwin: DATA.sites.darwin }).forEach(([name, s]) => {
  mg.append("circle").attr("cx", mx(s.lon)).attr("cy", my(s.lat)).attr("r", 5)
    .attr("fill", "#fff").attr("stroke", "#1f1e1c").attr("stroke-width", 2);
  mg.append("text").attr("x", mx(s.lon) + 8).attr("y", my(s.lat) + 4).attr("class", "boxlab").text(name);
});
const mhl = mg.append("rect").attr("width", cw).attr("height", ch).attr("fill", "none").attr("stroke", "#1f1e1c").style("display", "none");
mg.append("rect").attr("width", pw).attr("height", mapH).attr("fill", "transparent")
  .on("mousemove", ev => {
    const [px, py] = d3.pointer(ev);
    const j = Math.max(0, Math.min(nlon - 1, Math.floor(px / cw))), i = Math.max(0, Math.min(nlat - 1, Math.floor(py / ch)));
    const v = mp.diff[i][j];
    mhl.style("display", null).attr("x", j * cw).attr("y", i * ch);
    showTip(ev, `<b>${latFmt(lat[i])}, ${lonFmt(lon[j])}</b><br>El Niño minus La Niña: ${num(v, "hPa")}<br>` +
      (v == null ? "" : v > 0 ? "Pressure is <b>higher</b> during El Niño" : "Pressure is <b>lower</b> during El Niño"));
  })
  .on("mouseleave", () => { mhl.style("display", "none"); hideTip(); });
// map legend
const lg = svg.append("g").attr("transform", `translate(${ml},${mapTop + mapH + 40})`);
const grad = svg.append("defs").append("linearGradient").attr("id", "gAgrad");
d3.range(0, 1.0001, 0.05).forEach(t => grad.append("stop").attr("offset", t).attr("stop-color", pcolor(-L + 2 * L * t)));
lg.append("rect").attr("width", 260).attr("height", 10).attr("fill", "url(#gAgrad)");
lg.append("g").attr("class", "axis").attr("transform", "translate(0,10)")
  .call(d3.axisBottom(d3.scaleLinear([-L, L], [0, 260])).ticks(5).tickFormat(v => (v > 0 ? "+" : v < 0 ? "−" : "") + Math.abs(v)).tickSizeOuter(0));
lg.append("text").attr("x", 274).attr("y", 9).text("hPa. Orange = higher pressure during El Niño, purple = lower. Black lines = coastlines.");

// ---------------- stacked timelines ----------------
const T = DATA.time, N = T.length, yr = i => +T[i].slice(0, 4), mo = i => +T[i].slice(5, 7);
const S = M.soi_lim;
const x = d3.scaleLinear([0, N], [0, tw]);
const y1 = d3.scaleLinear([-3, 3], [th, 0]).clamp(true);
const y2 = d3.scaleLinear([-S, S], [th, 0]).clamp(true);
const tg = svg.append("g").attr("transform", `translate(${ml},${tTop})`);
const key = tg.append("g").attr("transform", `translate(0,-50)`);
[["El Niño", RED], ["La Niña", BLUE], ["Neutral", GRAY]].forEach(([n, c], k) => {
  key.append("rect").attr("x", k * 90).attr("width", 11).attr("height", 11).attr("fill", c);
  key.append("text").attr("x", k * 90 + 16).attr("y", 10).text(n);
});
key.append("text").attr("x", 280).attr("y", 10).text("Bar color = month's phase (same in both panels)");
const panels = [
  { y: y1, v: DATA.oni, title: "Ocean: Niño 3.4 index (°C)", off: 0, ticks: [-2, 0, 2] },
  { y: y2, v: DATA.soi, title: "Atmosphere: Tahiti minus Darwin pressure (hPa)", off: th + tgap, ticks: [-Math.floor(S / 2) * 1, 0, Math.floor(S / 2)] },
];
panels.forEach(p => {
  const g = tg.append("g").attr("transform", `translate(0,${p.off})`);
  g.append("text").attr("y", -8).attr("class", "ptitle").text(p.title);
  g.append("g").selectAll("rect").data(p.v).join("rect")
    .attr("x", (_, i) => x(i)).attr("width", Math.max(tw / N, 0.6))
    .attr("y", v => (v == null ? 0 : p.y(Math.max(v, 0)))).attr("height", v => (v == null ? 0 : Math.abs(p.y(v) - p.y(0))))
    .attr("fill", (_, i) => phaseColor(DATA.phase[i]));
  g.append("line").attr("x2", tw).attr("y1", p.y(0)).attr("y2", p.y(0)).attr("stroke", "#888780").attr("stroke-width", 0.6);
  g.append("g").attr("class", "axis").call(d3.axisLeft(p.y).tickValues(p.ticks).tickFormat(v => (v > 0 ? "+" : v < 0 ? "−" : "") + Math.abs(v)).tickSizeOuter(0));
});
const decades = d3.range(N).filter(i => mo(i) === 1 && yr(i) % 10 === 0);
tg.append("g").attr("class", "axis").attr("transform", `translate(0,${2 * th + tgap})`)
  .call(d3.axisBottom(x).tickValues(decades).tickFormat(i => yr(i)).tickSizeOuter(0));
const vline = tg.append("line").attr("y2", 2 * th + tgap).attr("stroke", "#1f1e1c").style("display", "none");
const brush = d3.brushX().extent([[0, 0], [tw, 2 * th + tgap]]).on("brush end", ev => {
  if (!ev.selection) { setSel(null); return; }
  const [a, b] = ev.selection.map(v => Math.max(0, Math.min(N, Math.round(x.invert(v)))));
  setSel(b - a >= 6 ? [a, b - 1] : null);
});
const gb = tg.append("g").attr("class", "brush").call(brush);
gb.on("mousemove.hover", ev => {
  const [px] = d3.pointer(ev, tg.node());
  const i = Math.max(0, Math.min(N - 1, Math.floor(x.invert(px))));
  vline.style("display", null).attr("x1", x(i + 0.5)).attr("x2", x(i + 0.5));
  ring.style("display", DATA.oni[i] == null || DATA.soi[i] == null ? "none" : null)
    .attr("cx", sx(DATA.oni[i] ?? 0)).attr("cy", sy(DATA.soi[i] ?? 0));
  showTip(ev, `<b>${T[i]}</b> · ${DATA.phase[i]}<br>Niño 3.4 index: ${num(DATA.oni[i], "°C")}<br>` +
    `Tahiti minus Darwin: ${num(DATA.soi[i], "hPa")}<br><span style="color:#6b6a65">Tahiti ${num(DATA.tahiti[i], "hPa")}, Darwin ${num(DATA.darwin[i], "hPa")} vs normal</span>`);
}).on("mouseleave.hover", () => { vline.style("display", "none"); ring.style("display", "none"); hideTip(); });

// ---------------- scatter ----------------
const sg = svg.append("g").attr("transform", `translate(${sLeft},${tTop})`);
const sx = d3.scaleLinear([-3, 3], [0, ss]).clamp(true), sy = d3.scaleLinear([-S, S], [ss, 0]).clamp(true);
sg.append("text").attr("y", -8).attr("class", "ptitle").text("Each month: ocean vs atmosphere");
sg.append("line").attr("x1", sx(0)).attr("x2", sx(0)).attr("y2", ss).attr("stroke", "#d3d1c7");
sg.append("line").attr("y1", sy(0)).attr("y2", sy(0)).attr("x2", ss).attr("stroke", "#d3d1c7");
const idx = d3.range(N).filter(i => DATA.oni[i] != null && DATA.soi[i] != null);
const dots = sg.append("g").selectAll("circle").data(idx).join("circle")
  .attr("cx", i => sx(DATA.oni[i])).attr("cy", i => sy(DATA.soi[i])).attr("r", 2.6)
  .attr("fill", i => phaseColor(DATA.phase[i]));
const ring = sg.append("circle").attr("r", 6).attr("fill", "none").attr("stroke", "#1f1e1c").attr("stroke-width", 1.5).style("display", "none");
sg.append("g").attr("class", "axis").attr("transform", `translate(0,${ss})`).call(d3.axisBottom(sx).ticks(5).tickSizeOuter(0));
sg.append("g").attr("class", "axis").call(d3.axisLeft(sy).ticks(5).tickSizeOuter(0));
sg.append("text").attr("x", ss / 2).attr("y", ss + 32).attr("text-anchor", "middle").text("Niño 3.4 index (°C) →  warmer ocean");
sg.append("text").attr("transform", `translate(-38,${ss / 2}) rotate(-90)`).attr("text-anchor", "middle").text("Tahiti − Darwin pressure (hPa)");
const rText = sg.append("text").attr("x", ss).attr("y", 14).attr("text-anchor", "end").attr("class", "boxlab");
const rSub = sg.append("text").attr("x", ss).attr("y", 28).attr("text-anchor", "end");

function corr(ids) {
  const a = ids.map(i => DATA.oni[i]), b = ids.map(i => DATA.soi[i]);
  const ma = d3.mean(a), mb = d3.mean(b);
  const num_ = d3.sum(a, (v, k) => (v - ma) * (b[k] - mb));
  return num_ / Math.sqrt(d3.sum(a, v => (v - ma) ** 2) * d3.sum(b, v => (v - mb) ** 2));
}
function setSel(w) {
  const ids = w ? idx.filter(i => i >= w[0] && i <= w[1]) : idx;
  dots.attr("opacity", i => (!w || (i >= w[0] && i <= w[1]) ? 0.8 : 0.08));
  rText.text(ids.length > 3 ? `r = ${corr(ids).toFixed(2)}` : "r = n/a");
  rSub.text(w ? `${T[w[0]]} to ${T[w[1]]} (${ids.length} months)` : `all ${ids.length} months`);
}
setSel(null);

root.append("p").attr("class", "note").html(
  "Hover the bars to read a month (its dot is circled on the right). <b>Drag across the bars</b> to pick a stretch of years; the scatter plot fades the other months and recalculates r. Click outside the shaded box to reset."
);

function showTip(ev, html) { tip.style("opacity", 1).style("left", (ev.pageX + 14) + "px").style("top", (ev.pageY + 10) + "px").html(html); }
function hideTip() { tip.style("opacity", 0); }
"""

show_d3("graphA_pressure_seesaw", "Graph A: the air-pressure seesaw", GRAPH_A_JS, DATA_A, height=980)

Saved out/graphA_pressure_seesaw.html; open it in a browser to see it full size.


### What to look for

1. **The seesaw on the map.** Is there one color over Australia and Indonesia and the opposite color over the central and southeastern Pacific? Where does the color change sign? Climate scientists call that line the seesaw's pivot.
2. **The mirror in the bars.** Find 1982–83 and 1997–98, the two strongest El Niños of the century. Big red bars up top should line up with big drops below. Then find La Niña 1988–89.
3. **How tight is the link?** Read *r* for all years, then drag to compare decades. A value around −0.7 or stronger means the atmosphere is closely locked to the ocean.

**Sentence for the video (fill in from the printed numbers):** "When the Pacific warms during El Niño, air pressure rises over Darwin and falls over Tahiti: the pressure difference averages ___ hPa in El Niño months versus ___ hPa in La Niña months, and month to month the two signals are correlated at r = ___."

## 4. Graph B: storm clouds on the move

### The idea

Over the warmest ocean, air rises and builds towering thunderstorms. Normally that is the western Pacific and Indonesia, a region scientists call the **warm pool**. When El Niño spreads warm water east, the storms should follow. That is why El Niño brings drought to Indonesia and Australia and floods to Peru and the central Pacific islands.

We can watch this happen using **outgoing longwave radiation** (`rlut`). Storm tops are very cold, so they send little heat to space: **low `rlut` = stormy**. We take a thin strip along the equator (5°S–5°N, from the Indian Ocean at 40°E to South America at 70°W), average it north to south, and stack one row per month into a **Hovmöller diagram**.

We also track one number per month: **how far east the storms reach**. That is the easternmost longitude between 100°E and 90°W where the 3-month average `rlut` is below **240 W/m²**, a common rule of thumb for deep storm clouds. During El Niño this edge should jump east.

### What the graph shows

| Part | What you see | How to read it |
|---|---|---|
| **Heatmap** | Rows = months (time runs **down**), columns = 2.5° slices of longitude, west (Indian Ocean) on the left, east (South America) on the right | *Change from normal* view: **teal = more storm clouds than usual**, **brown = clearer than usual**. *Actual value* view: **dark teal = stormy**, pale = clear. |
| **Black line** | The eastern edge of the stormy zone, month by month | When it swings right, storms have spread east across the Pacific. |
| **Bars on the right** | The Niño 3.4 index for the same months (red = El Niño) | Check that eastward swings of the line match red bars. |
| **Top strip** | The Niño 3.4 index for all 65 years | Drag across it to choose which years to show below. |

**Interaction:** drag on the top strip to pick a window of years (2–30); start with 1996–2000 to see the huge 1997–98 El Niño followed by La Niña. Toggle between *change from normal* (shows the shift) and *actual value* (shows where storms really are). Hover any row to read the date, the value at that longitude, the storm edge, and the ocean index.

**Design choices:** we use teal and brown instead of the red and blue of temperature, so readers don't mistake clouds for heat. Teal suggests wet and stormy, brown suggests dry and clear. The overview strip plus detail view lets readers scan 65 years for big events, then zoom into one. A single static picture of 780 months would be too crowded to read.

In [10]:
band = area_mean(rlut, dims=("lat",))                  # average 5°S–5°N, keep longitude → (time, lon)
band_anom = anomalies(band)

# Eastern edge of the stormy zone: easternmost longitude (100°E–270°E) with 3-month mean OLR < 240 W/m²
EDGE_THR, EDGE_LON = 240, (100, 270)
smooth = band.rolling(time=3, center=True).mean()
in_range = (smooth.lon >= EDGE_LON[0]) & (smooth.lon <= EDGE_LON[1])
stormy = (smooth < EDGE_THR) & in_range
edge = xr.where(stormy, smooth.lon, np.nan).max("lon")

step = float(np.diff(band.lon.values).mean())
factor = max(1, int(round(2.5 / step)))               # group into ~2.5° slices for the chart
anom_c = band_anom.coarsen(lon=factor, boundary="trim").mean().transpose("time", "lon")
tot_c = band.coarsen(lon=factor, boundary="trim").mean().transpose("time", "lon")

ok = ~np.isnan(edge.values) & ~np.isnan(oni.values)
r_edge = np.corrcoef(edge.values[ok], oni.values[ok])[0, 1]
lonlab = lambda l: "n/a" if np.isnan(l) else (f"{l:.0f}°E" if l <= 180 else f"{360 - l:.0f}°W")
print(f"Storm edge, average: El Niño months {lonlab(float(np.nanmean(edge.values[is_nino])))}, "
      f"La Niña months {lonlab(float(np.nanmean(edge.values[is_nina])))}, "
      f"neutral {lonlab(float(np.nanmean(edge.values[phase == 'Neutral'])))}")
print(f"Correlation between the storm edge and the ocean index: r = {r_edge:.2f}")
for d in ["1997-12", "1988-12", "2010-12"]:
    if d in dates:
        k = dates.index(d)
        print(f"  {d} ({phase[k]}): storms reach {lonlab(float(edge.values[k]))}")

Storm edge, average: El Niño months 166°W, La Niña months 154°E, neutral 168°E
Correlation between the storm edge and the ocean index: r = 0.62
  1997-12 (El Niño): storms reach 129°W
  1988-12 (La Niña): storms reach 155°E
  2010-12 (La Niña): storms reach 152°E


In [11]:
DATA_B = dict(
    meta={**META, "lon_step": round(step * factor, 2), "edge_thr": EDGE_THR,
          "anom_lim": float(max(10, np.ceil(np.nanpercentile(np.abs(anom_c), 98) / 5) * 5)),
          "total_min": float(np.floor(np.nanpercentile(tot_c, 1) / 10) * 10),
          "total_max": float(np.ceil(np.nanpercentile(tot_c, 99) / 10) * 10)},
    lon=clean(anom_c.lon.values), time=dates, anom=clean(anom_c.values, 1), total=clean(tot_c.values, 0),
    edge=clean(edge.values, 1), oni=clean(oni.values), phase=list(phase),
    default=[dates.index("1980-01"), dates.index("1999-12")],
)
(OUT / "data" / "olr_hovmoller.json").write_text(json.dumps(DATA_B, allow_nan=False))

1034784

In [12]:
GRAPH_B_JS = r"""
// Graph B: storm clouds along the equator (outgoing longwave radiation Hovmöller) + Niño 3.4 index (D3 v7)
// DATA = {meta, lon[], time[], anom[[...]], total[[...]], edge[], oni[], phase[], default:[i0,i1]}
const M = DATA.meta;
const root = d3.select("#root");
root.append("h2").text("During El Niño, the Pacific's storm clouds march east");
root.append("p").attr("class", "sub").html(
  `Each row is one month (time runs downward) and each column is a ${M.lon_step}° slice of longitude along the equator (average of 5°S–5°N), ` +
  `from the Indian Ocean on the left to South America on the right. Color = <b>outgoing longwave radiation</b>, the heat Earth sends to space. ` +
  `Tall storm clouds have very cold tops and send out little heat, so <b>low values mean storms</b>. ` +
  `The black line marks the eastern edge of the stormy zone (3-month average below ${M.edge_thr} W/m²). ` +
  `${M.model} ${M.experiment} run (${M.member}): real observed ocean temperatures, simulated clouds.`
);
const ctr = root.append("div").attr("class", "controls");
ctr.append("span").style("font-weight", "600").text("Show:");
let view = "anom", showEdge = true;
[["anom", "Change from normal"], ["total", "Actual value"]].forEach(([k, label]) => {
  const l = ctr.append("label");
  l.append("input").attr("type", "radio").attr("name", "view").property("checked", k === view)
    .on("change", () => { view = k; render(); drawLegend(); });
  l.append("span").text(" " + label);
});
const el = ctr.append("label");
el.append("input").attr("type", "checkbox").property("checked", true).on("change", ev => { showEdge = ev.target.checked; render(); });
el.append("span").text(" Eastern edge of storms");
root.append("div").attr("class", "controls")
  .html("<span style='font-weight:600'>Choose years:</span> drag across the strip below (2 to 30 years); drag the shaded box to move it.");

const T = DATA.time, N = T.length, lon = DATA.lon, nlon = lon.length, dlon = lon[1] - lon[0];
const yr = i => +T[i].slice(0, 4), mo = i => +T[i].slice(5, 7);
const LA = M.anom_lim;
const anomColor = d3.scaleDiverging([-LA, 0, LA],
  d3.piecewise(d3.interpolateLab, ["#0f4f4b", "#2a8c86", "#9fd3cf", "#f0efec", "#e3c79a", "#b0753a", "#6b3d12"])).clamp(true);
const totColor = d3.scaleSequential([M.total_min, M.total_max],
  d3.piecewise(d3.interpolateLab, ["#0b3b38", "#2a8c86", "#9fd3cf", "#efe6d2"])).clamp(true);
const RED = "#e34948", BLUE = "#2a78d6", GRAY = "#c9c6bd";
const phaseColor = p => (p === "El Niño" ? RED : p === "La Niña" ? BLUE : GRAY);
const num = (v, u, d = 1) => (v == null ? "n/a" : (v >= 0 ? "+" : "−") + Math.abs(v).toFixed(d) + " " + u);
const lonFmt = l => { l = Math.round(((l % 360) + 360) % 360); return l === 180 ? "180°" : l < 180 ? `${l}°E` : `${360 - l}°W`; };

const W = 880, ml = 52, mr = 20, ow = W - ml - mr, oh = 54;
const mainTop = 34 + oh + 50, mw = 580, sgap = 28, sw = W - ml - mr - mw - sgap, mh = 520;
const H = mainTop + mh + 104;
const svg = root.append("svg").attr("viewBox", `0 0 ${W} ${H}`).attr("width", W).style("max-width", "100%").style("height", "auto");
const tip = d3.select("body").append("div").attr("class", "tip");

// overview strip
const og = svg.append("g").attr("transform", `translate(${ml},34)`);
og.append("text").attr("y", -12).attr("class", "ptitle").text(`Niño 3.4 index, every month ${yr(0)}–${yr(N - 1)} (pick years here)`);
const xo = d3.scaleLinear([0, N], [0, ow]);
const yo = d3.scaleLinear([-3, 3], [oh, 0]).clamp(true);
og.append("g").selectAll("rect").data(DATA.oni).join("rect")
  .attr("x", (_, i) => xo(i)).attr("width", Math.max(ow / N, 0.6))
  .attr("y", v => (v == null ? 0 : yo(Math.max(v, 0)))).attr("height", v => (v == null ? 0 : Math.abs(yo(v) - yo(0))))
  .attr("fill", (_, i) => phaseColor(DATA.phase[i]));
og.append("line").attr("x2", ow).attr("y1", yo(0)).attr("y2", yo(0)).attr("stroke", "#888780").attr("stroke-width", 0.6);
og.append("g").attr("class", "axis").attr("transform", `translate(0,${oh})`)
  .call(d3.axisBottom(xo).tickValues(d3.range(N).filter(i => mo(i) === 1 && yr(i) % 10 === 0)).tickFormat(i => yr(i)).tickSizeOuter(0));
const brush = d3.brushX().extent([[0, 0], [ow, oh]]).on("brush end", brushed);
const gb = og.append("g").attr("class", "brush").call(brush);

// main panel + side panel
const mg = svg.append("g").attr("transform", `translate(${ml},${mainTop})`);
const sg = svg.append("g").attr("transform", `translate(${ml + mw + sgap},${mainTop})`);
const xm = d3.scaleLinear([lon[0] - dlon / 2, lon[nlon - 1] + dlon / 2], [0, mw]);
const xs = d3.scaleLinear([-3, 3], [0, sw]).clamp(true);
let y = d3.scaleLinear([0, 1], [0, mh]);
let win = DATA.default.slice();
const mTitle = mg.append("text").attr("y", -10).attr("class", "ptitle");
sg.append("text").attr("y", -10).attr("class", "ptitle").text("Niño 3.4 index");
const img = mg.append("image").attr("width", mw).attr("height", mh).attr("preserveAspectRatio", "none").style("image-rendering", "pixelated");
const gridG = mg.append("g"), sideGridG = sg.append("g");
const edgePath = mg.append("path").attr("fill", "none").attr("stroke", "#1f1e1c").attr("stroke-width", 1.6);
const lonTicks = [60, 90, 120, 150, 180, 210, 240, 270].filter(t => t > xm.domain()[0] && t < xm.domain()[1]);
mg.append("g").attr("class", "axis").attr("transform", `translate(0,${mh})`).call(d3.axisBottom(xm).tickValues(lonTicks).tickFormat(lonFmt).tickSizeOuter(0));
[["Indian Ocean", 75], ["Indonesia", 118], ["Pacific Ocean", 200]].forEach(([n, l]) => {
  if (l > xm.domain()[0] && l < xm.domain()[1])
    mg.append("text").attr("x", xm(l)).attr("y", mh + 34).attr("text-anchor", "middle").attr("class", "boxlab").text(n);
});
const yAxisG = mg.append("g").attr("class", "axis");
const barsG = sg.append("g");
[-0.5, 0.5].forEach(v => sg.append("line").attr("x1", xs(v)).attr("x2", xs(v)).attr("y2", mh).attr("stroke", "#888780").attr("stroke-dasharray", "3 3"));
sg.append("line").attr("x1", xs(0)).attr("x2", xs(0)).attr("y2", mh).attr("stroke", "#5f5e5a");
sg.append("g").attr("class", "axis").attr("transform", `translate(0,${mh})`)
  .call(d3.axisBottom(xs).tickValues([-2, 0, 2]).tickFormat(v => (v > 0 ? "+" : v < 0 ? "−" : "") + Math.abs(v)).tickSizeOuter(0));
sg.append("text").attr("x", sw / 2).attr("y", mh + 32).attr("text-anchor", "middle").text("°C (red = El Niño)");

const guide = svg.append("line").attr("stroke", "#1f1e1c").attr("stroke-width", 1).style("display", "none").style("pointer-events", "none");
mg.append("rect").attr("width", mw).attr("height", mh).attr("fill", "transparent").on("mousemove", ev => hover(ev, mg, true)).on("mouseleave", leave);
sg.append("rect").attr("width", sw).attr("height", mh).attr("fill", "transparent").on("mousemove", ev => hover(ev, sg, false)).on("mouseleave", leave);

const lg = svg.append("g").attr("transform", `translate(${ml},${mainTop + mh + 66})`);
const defs = svg.append("defs");
function drawLegend() {
  lg.selectAll("*").remove(); defs.selectAll("*").remove();
  const sc = view === "anom" ? anomColor : totColor, dom = view === "anom" ? [-LA, LA] : [M.total_min, M.total_max];
  const grad = defs.append("linearGradient").attr("id", "gBgrad");
  d3.range(0, 1.0001, 0.05).forEach(t => grad.append("stop").attr("offset", t).attr("stop-color", sc(dom[0] + (dom[1] - dom[0]) * t)));
  lg.append("rect").attr("width", 300).attr("height", 10).attr("fill", "url(#gBgrad)");
  lg.append("g").attr("class", "axis").attr("transform", "translate(0,10)")
    .call(d3.axisBottom(d3.scaleLinear(dom, [0, 300])).ticks(6).tickSizeOuter(0));
  lg.append("text").attr("x", 314).attr("y", 9).text(view === "anom"
    ? "W/m² vs normal: teal = more storm clouds than usual, brown = clearer skies"
    : "W/m²: dark teal = stormy (cold cloud tops), pale = clear skies");
}

const canvas = document.createElement("canvas");
function render() {
  const [i0, i1] = win, rows = i1 - i0 + 1, src = view === "anom" ? DATA.anom : DATA.total, sc = view === "anom" ? anomColor : totColor;
  mTitle.text(view === "anom" ? "Outgoing longwave radiation: change from normal" : "Outgoing longwave radiation: actual value");
  y = d3.scaleLinear([i0, i1 + 1], [0, mh]);
  canvas.width = nlon; canvas.height = rows;
  const ctx = canvas.getContext("2d"), im = ctx.createImageData(nlon, rows);
  for (let r = 0; r < rows; r++) for (let c = 0; c < nlon; c++) {
    const v = src[i0 + r][c], col = d3.rgb(v == null ? "#f6f5f2" : sc(v)), k = 4 * (r * nlon + c);
    im.data[k] = col.r; im.data[k + 1] = col.g; im.data[k + 2] = col.b; im.data[k + 3] = 255;
  }
  ctx.putImageData(im, 0, 0);
  img.attr("href", canvas.toDataURL());
  const span = rows / 12, step = span <= 12 ? 1 : span <= 24 ? 2 : 5;
  const ticks = d3.range(i0, i1 + 1).filter(i => mo(i) === 1 && yr(i) % step === 0);
  yAxisG.call(d3.axisLeft(y).tickValues(ticks).tickFormat(i => yr(i)).tickSizeOuter(0));
  [gridG, sideGridG].forEach((g, k) => g.selectAll("line").data(ticks).join("line")
    .attr("x2", k ? sw : mw).attr("y1", i => y(i)).attr("y2", i => y(i))
    .attr("stroke", k ? "#e4e2dc" : "rgba(255,255,255,0.35)").attr("stroke-width", 0.6));
  const bh = mh / rows, ids = d3.range(i0, i1 + 1);
  edgePath.style("display", showEdge ? null : "none")
    .attr("d", d3.line().defined(i => DATA.edge[i] != null).x(i => xm(DATA.edge[i])).y(i => y(i) + bh / 2)(ids));
  barsG.selectAll("rect").data(ids).join("rect")
    .attr("y", i => y(i)).attr("height", Math.max(bh - (bh > 3 ? 1 : 0), 0.5))
    .attr("x", i => (DATA.oni[i] == null ? xs(0) : xs(Math.min(DATA.oni[i], 0))))
    .attr("width", i => (DATA.oni[i] == null ? 0 : Math.abs(xs(DATA.oni[i]) - xs(0))))
    .attr("fill", i => phaseColor(DATA.phase[i]));
}
function brushed(ev) {
  if (!ev.selection) { if (ev.type === "end") gb.call(brush.move, [xo(DATA.default[0]), xo(DATA.default[1] + 1)]); return; }
  let a = Math.round(xo.invert(ev.selection[0])), b = Math.round(xo.invert(ev.selection[1]));
  const len = Math.max(24, Math.min(360, b - a));
  if (ev.type === "end" && ev.sourceEvent && len !== b - a) { if (a + len > N) a = N - len; gb.call(brush.move, [xo(a), xo(a + len)]); return; }
  win = [a, Math.min(N, a + len) - 1];
  render();
}
function hover(ev, g, isMain) {
  const [mx, my] = d3.pointer(ev, g.node());
  const i = Math.max(win[0], Math.min(win[1], Math.floor(y.invert(my))));
  const gy = mainTop + y(i) + (y(i + 1) - y(i)) / 2;
  guide.style("display", null).attr("x1", ml).attr("x2", ml + mw + sgap + sw).attr("y1", gy).attr("y2", gy);
  let html = `<b>${T[i]}</b> · ${DATA.phase[i]}`;
  if (isMain) {
    const c = Math.max(0, Math.min(nlon - 1, Math.floor((mx / mw) * nlon)));
    html += `<br>${lonFmt(lon[c])}: ${DATA.total[i][c] == null ? "n/a" : DATA.total[i][c].toFixed(0) + " W/m²"} ` +
      `(${num(DATA.anom[i][c], "vs normal")})`;
  }
  html += `<br>Storms reach east to: ${DATA.edge[i] == null ? "n/a" : lonFmt(DATA.edge[i])}`;
  html += `<br>Niño 3.4 index: ${num(DATA.oni[i], "°C", 2)}`;
  tip.style("opacity", 1).style("left", (ev.pageX + 14) + "px").style("top", (ev.pageY + 10) + "px").html(html);
}
function leave() { tip.style("opacity", 0); guide.style("display", "none"); }

drawLegend();
gb.call(brush.move, [xo(win[0]), xo(win[1] + 1)]);
root.append("p").attr("class", "note").text("Hover the heatmap or the bars to read one month. Years are real calendar years: amip feeds the model the ocean temperatures that were actually observed.");
"""

show_d3("graphB_storm_clouds", "Graph B: storm clouds on the move", GRAPH_B_JS, DATA_B, height=1030)

Saved out/graphB_storm_clouds.html; open it in a browser to see it full size.


### What to look for

1. **The warm pool.** In *actual value* view, a dark teal band should sit over Indonesia and the western Pacific almost all the time. That is the normal home of the storms.
2. **El Niño's march east.** Drag to 1996–2000. During 1997–98 the teal (*change from normal* view) should spread toward the central and eastern Pacific, with brown (clearer skies) over Indonesia, and the black line should swing far right. During the La Niña that follows, it should pull back west.
3. **Does it always happen?** Compare a few El Niños. Do weak ones move the storms much? This links to our team's question about strength.

**Sentence for the video (fill in from the printed numbers):** "Storm clouds follow the warm water: in La Niña months they reach only about ___, but in El Niño months they extend to ___, and during the 1997–98 El Niño they reached ___. That shift is why El Niño dries out Indonesia and soaks the central Pacific."

## 5. Files produced

| File | What it is |
|---|---|
| `out/graphA_pressure_seesaw.html` | Graph A as a standalone page (open in any browser) |
| `out/graphB_storm_clouds.html` | Graph B as a standalone page |
| `out/data/soi.csv` | Monthly Niño 3.4 index, Tahiti−Darwin pressure, and phase |
| `out/data/olr_hovmoller.json` | All data behind Graph B, reusable on our D3 site |

### Data transformations (for our write-up)

The project rubric requires every transformation to be described to the reader. For these two graphs:

1. **Units:** pressure Pa → hPa; surface temperature K → °C.
2. **Anomalies:** every variable is shown relative to its 1981–2010 average for the same calendar month.
3. **Smoothing:** the ocean index and the Tahiti−Darwin difference are 3-month running means.
4. **El Niño / La Niña months:** ocean index beyond ±0.5 °C for at least 5 consecutive months (NOAA's rule).
5. **Graph A map:** average of all El Niño months minus average of all La Niña months, on ~2° boxes.
6. **Graph B:** averaged 5°S–5°N and grouped into 2.5° longitude slices. Storm edge = easternmost point between 100°E and 90°W where the 3-month mean is below 240 W/m².
7. **Stations:** Tahiti and Darwin are represented by the nearest model grid box (~1° wide), not the actual weather stations.